In [27]:
import torch
import numpy
import soundfile as sf

from wavenet.data import DATA_DIR, AudioChunks
from wavenet.model import WaveNet
from torch.utils.data import DataLoader
import torch.nn.functional as F
from pathlib import Path

device = "mps" if torch.backends.mps.is_available() else "cpu"

In [28]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [29]:
train_data = torch.load(DATA_DIR / "train_8000.pt")
val_data   = torch.load(DATA_DIR / "val_8000.pt")

In [30]:
model = WaveNet(R=64, S=128).to(device = device)
opt = torch.optim.AdamW(model.parameters(), lr = 1e-3)

In [31]:
T, B = 4096, 8
train_dl = DataLoader(AudioChunks(train_data, T, model.rf, n_items=10_000), batch_size=B)
val_dl   = DataLoader(AudioChunks(val_data, T, model.rf, random=False), batch_size=B)

In [32]:
@torch.no_grad()
def evaluate(max_batches=50):
    model.eval()
    losses = []
    for i, (x, y) in enumerate(val_dl):
        if i == max_batches:
            break
        x = x.to(device)
        y = y.to(device)
        
        logits = model(x)
        loss = F.cross_entropy(logits, y, ignore_index=-1)
        
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)

In [33]:
max_steps, eval_every = 500, 250
step, best_val = 0, float('inf')

while step < max_steps:
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        
        # forward
        logits = model(x, c=None)
        loss = F.cross_entropy(logits, y, ignore_index=-1)
        
        # backward
        opt.zero_grad()
        loss.backward()
        opt.step()
        
        if step % eval_every == 0:
            val = evaluate()
            print(f"step {step}: train {loss.item():.3f}  val {val:.3f}")
            
            if val < best_val:
                best_val = val
                torch.save({"model": model.state_dict(), "step": step, "val": val}, "../checkpoints/ckpt.pt")
        
        step += 1
        if step >= max_steps:
            break

KeyboardInterrupt: 

In [ ]:
loss.item()

3.672468900680542

## Mel Conditioning WaveNet

In [34]:
import torch

torch.manual_seed(0)
B, T, F_ = 4, 4096, 4096 // 128
x = torch.randint(0, 256, (B, T))
c = torch.randn(B, 80, F_)

# 1. unconditional still works
m0 = WaveNet(R=64, S=128)
assert m0(x).shape == (B, 256, T)

# 2. conditional works + param count
m1 = WaveNet(R=64, S=128, n_mels=80, hop=128)
assert m1(x, c).shape == (B, 256, T)
print(f"params: uncond {sum(p.numel() for p in m0.parameters()):,}  "
      f"cond {sum(p.numel() for p in m1.parameters()):,}")   # cond ≈ +185k

# 3. mismatches fail loudly
for model, args in [(m1, (x,)), (m0, (x, c))]:
    try:
        model(*args); raise AssertionError("should have raised")
    except ValueError:
        pass

# 4. conditioning is actually used
assert not torch.allclose(m1(x, c), m1(x, c + 1.0))

# 5. audio causality with c held fixed
captured = {}
def hook(mod, inp, out):
    out.retain_grad()
    captured["e"] = out          # no return -> output left unchanged
h = m1.embed.register_forward_hook(hook)
t = 3000
m1(x, c)[:, :, t].sum().backward()
idx = captured["e"].grad.abs().sum(dim=(0, 2)).nonzero().flatten()
h.remove()
print(f"t={t}: inputs {idx.min().item()}..{idx.max().item()}, count {len(idx)}")
assert idx.min() == t - m1.rf + 1 and idx.max() == t and len(idx) == m1.rf

print("all conditional-model tests passed")

params: uncond 596,032  cond 782,656
t=3000: inputs 1977..3000, count 1024
all conditional-model tests passed
